# 04 - Analise estatistica e por regime

Carrega o CSV bruto gerado por `src/pipeline/run_all.py` e produz:
1. Tabela-resumo de metricas por modelo.
2. Diagrama de diferenca critica (Demsar 2006) via `autorank`.
3. Analise Bayesiana par a par com ROPE via `baycomp`.
4. Quebra dos resultados por regime (tamanho, numero de classes, proporcao categorica, missing).

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

import pandas as pd

from data.load_tabarena import summarize, RECOMMENDED_TASK_IDS
from src.reports.results_table import summary_by_model, pivot_for_stats
from src.pipeline.stats import demsar_analysis, bayesian_pairwise
from src.pipeline.regime import assign_regimes, aggregate_by_regime

In [2]:
raw = pd.read_csv('../results/raw.csv')
raw.head()

FileNotFoundError: [Errno 2] No such file or directory: '../results/raw.csv'

In [3]:
summary_by_model(raw)

,model,auc_ovo_mean,auc_ovo_std,accuracy_mean,accuracy_std,g_mean_mean,g_mean_std,cross_entropy_mean,cross_entropy_std,total_time_s_mean,total_time_s_std
0,catboost,0.957070,0.049858,0.838165,0.110783,0.600729,0.419896,0.513090,0.335337,4.092334,4.524277
2,xgboost,0.948975,0.062330,0.837110,0.096641,0.710190,0.247500,0.727173,0.482845,4.929101,2.453599
1,lightgbm,0.943590,0.073251,0.842332,0.105408,0.610313,0.423758,0.688898,0.447626,9.934438,6.786485


In [4]:
pivot = pivot_for_stats(raw, metric='auc_ovo')
demsar = demsar_analysis(pivot, output_dir=Path('../results/figures'))
demsar['ranking']

ValueError: requires at least five performance estimations (i.e., rows)

In [8]:
bayes = bayesian_pairwise(pivot, rope=0.01)
bayes

,model_a,model_b,p_a_worse,p_equivalent,p_a_better
0,catboost,group_model,0.00000,0.15626,0.84374
1,catboost,lightgbm,0.84298,0.15702,0.00000
2,catboost,xgboost,0.84130,0.15870,0.00000
3,group_model,lightgbm,0.84282,0.15718,0.00000
4,group_model,xgboost,0.84160,0.15840,0.00000
5,lightgbm,xgboost,0.00000,0.15812,0.84188


In [9]:
metadata = assign_regimes(summarize(RECOMMENDED_TASK_IDS))
for col in ['regime_size', 'regime_classes', 'regime_cat_share', 'regime_missing']:
    print(f'\n=== Agregado por {col} ===')
    print(aggregate_by_regime(raw, metadata, regime_col=col, metric_col='auc_ovo'))


=== Agregado por regime_size ===
  regime_size        model      mean  std  count
0       small     catboost  0.885668  NaN      1
1       small  group_model  0.999617  NaN      1
2       small     lightgbm  0.836150  NaN      1
3       small      xgboost  0.858282  NaN      1

=== Agregado por regime_classes ===
  regime_classes        model      mean  std  count
0     multiclass     catboost  0.885668  NaN      1
1     multiclass  group_model  0.999617  NaN      1
2     multiclass     lightgbm  0.836150  NaN      1
3     multiclass      xgboost  0.858282  NaN      1

=== Agregado por regime_cat_share ===
  regime_cat_share        model      mean  std  count
0              low     catboost  0.885668  NaN      1
1              low  group_model  0.999617  NaN      1
2              low     lightgbm  0.836150  NaN      1
3              low      xgboost  0.858282  NaN      1

=== Agregado por regime_missing ===
  regime_missing        model      mean  std  count
0             no     catbo